# 11001 ROW V2 Full Rebuild: TXT Rehash, Row Parse, SW1-SW4 Decode, Aircraft Pairing

This notebook deliberately rebuilds the canonical `ECFR_11001_ROW_V2` dataset in place.
It recalculates SHA-256 over every TXT file under the TXT root, preserves every physical occurrence, parses each unique byte-content hash once, decodes all model-applicable Status Words 1-4, and embeds aircraft identity and conservative cross-side engine pairing into the canonical Parquet records.

**Important:** Cell 02 deletes the existing V2 records/checkpoints/reports after explicit configuration checks. Original TXT files are never modified.

In [11]:
# OPTIONAL CELL 00 - CLEAN GENERATED V2 DATA BEFORE A COMPLETELY FRESH START
# DO NOT RUN DURING A RESUME. Uncomment only when intentionally discarding all
# generated ROW V2 records, checkpoints, reports, and inventories.
# Original TXT files under All_ECFRs_working/txt are not deleted.
#
#from pathlib import Path
#import shutil
#
#work_root = Path('/raid3/e296408/All_ECFRs_working')
#output_root = work_root / 'ECFR_11001_ROW_V2'
#expected = (work_root / 'ECFR_11001_ROW_V2').resolve()
#if output_root.resolve() != expected:
#    raise RuntimeError(f'Refusing unexpected cleanup path: {output_root}')
#for name in ['records', 'checkpoints', 'reports', 'inventory']:
#    target = output_root / name
#    if target.exists():
#        print('Deleting:', target)
#        shutil.rmtree(target)
#print('Optional cleanup complete. Source TXT files were not modified.')


In [12]:
# CELL 01 - Configuration and safety contract
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
import multiprocessing as mp
import os, re, json, math, hashlib, shutil, traceback
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

WORK_ROOT = Path('/raid3/e296408/All_ECFRs_working')
TXT_ROOT = WORK_ROOT / 'txt'
OUTPUT_ROOT = WORK_ROOT / 'ECFR_11001_ROW_V2'
RECORD_ROOT = OUTPUT_ROOT / 'records'
CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
REPORT_ROOT = OUTPUT_ROOT / 'reports'
INVENTORY_ROOT = OUTPUT_ROOT / 'inventory'

DECODER_CANDIDATES = [
    Path('/raid3/e296408/Notebooks/converted_data.xlsx'),
    Path('/raid3/e296408/converted_data.xlsx'),
    Path('/raid3/E296408/Converted data.xls'),
    WORK_ROOT / 'converted_data.xlsx',
]
DECODER_XLSX = next((p for p in DECODER_CANDIDATES if p.is_file()), None)
if DECODER_XLSX is None:
    raise FileNotFoundError('No decoder workbook found in DECODER_CANDIDATES')
if not TXT_ROOT.is_dir():
    raise FileNotFoundError(f'TXT_ROOT does not exist: {TXT_ROOT}')

# Set INITIALIZE_FULL_REBUILD=True only for the first clean start.
# After Cell 02 succeeds, set it False before any kernel restart/rerun.
INITIALIZE_FULL_REBUILD = True
RESUME_REBUILD = False
FULL_REBUILD = True
CONFIRM_FULL_REBUILD = 'REBUILD_ECFR_11001_ROW_V2'
REQUIRED_CONFIRMATION = 'REBUILD_ECFR_11001_ROW_V2'
PILOT_MODE = False
PILOT_UNIQUE_FILES = 1000
MAX_UNIQUE_FILES = PILOT_UNIQUE_FILES if PILOT_MODE else None
HASH_WORKERS = max(10, min(32, os.cpu_count() or 10))
MAX_WORKERS = max(10, min(60, os.cpu_count() or 10))
BATCH_SIZE = 500
PRINT_EVERY = 100
TIMESTAMP_CONTRACT_VERSION = 'SECTION_GMT_YYMMDD_HHMMSS_V1'
PARSER_VERSION = '11001-row-v2-section-time-20260920'
DECODER_VERSION = '11001-sw1-sw4-full-20260920'
RUN_ID = '11001_ROW_V2_TIMESTAMP_CORRECTED_20260920'
MIN_VALID_TIMESTAMP = pd.Timestamp('2000-01-01 00:00:00', tz='UTC')
MAX_VALID_TIMESTAMP = pd.Timestamp.now(tz='UTC')

TXT_OCCURRENCE_INVENTORY = INVENTORY_ROOT / 'txt_occurrence_inventory.parquet'
TXT_UNIQUE_INVENTORY = INVENTORY_ROOT / 'txt_unique_content_inventory.parquet'
MANIFEST_PATH = CHECKPOINT_ROOT / 'pipeline_manifest.parquet'
PAIRING_MANIFEST_PATH = CHECKPOINT_ROOT / 'aircraft_pairing_manifest.parquet'
HASH_CHECKPOINT_ROOT = CHECKPOINT_ROOT / 'hash_chunks'
HASH_PROGRESS_PATH = CHECKPOINT_ROOT / 'hash_progress.parquet'

for name in ['OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS','ARROW_NUM_THREADS']:
    os.environ[name] = '1'

print({
    'run_id': RUN_ID,
    'txt_root': str(TXT_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'decoder': str(DECODER_XLSX),
    'full_rebuild': FULL_REBUILD,
    'initialize_full_rebuild': INITIALIZE_FULL_REBUILD,
    'resume_rebuild': RESUME_REBUILD,
    'timestamp_contract_version': TIMESTAMP_CONTRACT_VERSION,
    'hash_workers': HASH_WORKERS,
    'processing_workers': MAX_WORKERS,
    'batch_size': BATCH_SIZE,
    'pilot_mode': PILOT_MODE,
    'max_unique_files': MAX_UNIQUE_FILES,
})


{'run_id': '11001_ROW_V2_TIMESTAMP_CORRECTED_20260920', 'txt_root': '/raid3/e296408/All_ECFRs_working/txt', 'output_root': '/raid3/e296408/All_ECFRs_working/ECFR_11001_ROW_V2', 'decoder': '/raid3/e296408/Notebooks/converted_data.xlsx', 'full_rebuild': True, 'initialize_full_rebuild': True, 'resume_rebuild': False, 'timestamp_contract_version': 'SECTION_GMT_YYMMDD_HHMMSS_V1', 'hash_workers': 32, 'processing_workers': 60, 'batch_size': 500, 'pilot_mode': False, 'max_unique_files': None}


In [13]:
# CELL 02 - Initialize once, or preserve checkpoints for restart
if not FULL_REBUILD:
    raise RuntimeError('This notebook is the full-rebuild notebook. Set FULL_REBUILD=True.')
if INITIALIZE_FULL_REBUILD and RESUME_REBUILD:
    raise RuntimeError('Choose initialization or resume, not both.')
if not INITIALIZE_FULL_REBUILD and not RESUME_REBUILD:
    raise RuntimeError('Set either INITIALIZE_FULL_REBUILD=True or RESUME_REBUILD=True.')
if CONFIRM_FULL_REBUILD != REQUIRED_CONFIRMATION:
    raise RuntimeError('Full rebuild confirmation string does not match.')
if OUTPUT_ROOT.resolve() != (WORK_ROOT / 'ECFR_11001_ROW_V2').resolve():
    raise RuntimeError(f'Refusing unexpected output path: {OUTPUT_ROOT}')
if TXT_ROOT.resolve() == OUTPUT_ROOT.resolve() or TXT_ROOT.resolve().is_relative_to(OUTPUT_ROOT.resolve()):
    raise RuntimeError('TXT_ROOT must not be inside OUTPUT_ROOT.')

if INITIALIZE_FULL_REBUILD:
    for path in [RECORD_ROOT, CHECKPOINT_ROOT, REPORT_ROOT, INVENTORY_ROOT]:
        if path.exists(): shutil.rmtree(path)
        path.mkdir(parents=True, exist_ok=True)
    print('FULL REBUILD INITIALIZED FROM ZERO')
else:
    for path in [OUTPUT_ROOT, RECORD_ROOT, CHECKPOINT_ROOT, REPORT_ROOT, INVENTORY_ROOT]:
        path.mkdir(parents=True, exist_ok=True)
    print('RESUME MODE: existing records and checkpoints preserved')
HASH_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print('Original TXT files remain untouched:', TXT_ROOT)


FULL REBUILD INITIALIZED FROM ZERO
Original TXT files remain untouched: /raid3/e296408/All_ECFRs_working/txt


In [14]:
# CELL 03 - Restartable full SHA-256 scan of every TXT physical path
# Checkpoint identity uses path + size + mtime only to determine whether a
# previously calculated hash can be reused during this same rebuild. A changed
# file is rehashed. Final duplicates are determined only by full-byte SHA-256.
def file_sha256(path, chunk_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(chunk_size),b''):digest.update(block)
    return digest.hexdigest()
def path_signature(path):
    st=Path(path).stat();return f'{Path(path).resolve()}|{st.st_size}|{st.st_mtime_ns}'
def hash_txt(path):
    path=Path(path)
    try:
        st=path.stat();return {'txt_file':str(path),'path_signature':path_signature(path),'txt_sha256':file_sha256(path),'txt_size_bytes':int(st.st_size),'txt_mtime_ns_reference_only':int(st.st_mtime_ns),'hash_status':'PASS','hash_error':''}
    except Exception as exc:return {'txt_file':str(path),'path_signature':None,'txt_sha256':None,'txt_size_bytes':None,'txt_mtime_ns_reference_only':None,'hash_status':'FAIL','hash_error':f'{type(exc).__name__}: {exc}'}

txt_files=sorted(set(TXT_ROOT.rglob('*.txt'))|set(TXT_ROOT.rglob('*.TXT')))
if not txt_files:raise RuntimeError(f'No TXT files found under {TXT_ROOT}')
current=pd.DataFrame({'txt_file':[str(p) for p in txt_files]})
current['path_signature']=current.txt_file.map(path_signature)
prior_hash=pd.read_parquet(HASH_PROGRESS_PATH) if HASH_PROGRESS_PATH.exists() else pd.DataFrame()
if not prior_hash.empty and {'path_signature','hash_status'}.issubset(prior_hash.columns):
    reusable=prior_hash.loc[prior_hash.hash_status.eq('PASS')].drop_duplicates('path_signature',keep='last')
    reuse_map=reusable.set_index('path_signature').to_dict('index')
else:reuse_map={}
pending=[p for p,s in zip(txt_files,current.path_signature) if s not in reuse_map]
reused=[dict(reuse_map[s],txt_file=str(p),path_signature=s) for p,s in zip(txt_files,current.path_signature) if s in reuse_map]
print({'physical_txt_paths':len(txt_files),'hashes_reused_from_checkpoint':len(reused),'hashes_pending':len(pending),'hash_workers':HASH_WORKERS})
rows=list(reused);new_rows=[]
with ThreadPoolExecutor(max_workers=HASH_WORKERS) as pool:
    futures={pool.submit(hash_txt,p):p for p in pending}
    for idx,future in enumerate(as_completed(futures),1):
        r=future.result();rows.append(r);new_rows.append(r)
        if idx%1000==0 or idx==len(futures):
            progress=pd.DataFrame(rows).drop_duplicates('path_signature',keep='last')
            tmp=HASH_PROGRESS_PATH.with_suffix('.tmp.parquet');progress.to_parquet(tmp,index=False,compression='zstd');tmp.replace(HASH_PROGRESS_PATH)
            print(f'Hashed/checkpointed {idx:,}/{len(futures):,}')
inventory=pd.DataFrame(rows).sort_values(['txt_sha256','txt_file'],na_position='last').reset_index(drop=True)
inventory['occurrence_order']=inventory.groupby('txt_sha256',dropna=False).cumcount()
inventory['is_representative']=inventory.hash_status.eq('PASS')&inventory.occurrence_order.eq(0)
atomic_tmp=TXT_OCCURRENCE_INVENTORY.with_suffix('.tmp.parquet');inventory.to_parquet(atomic_tmp,index=False,compression='zstd');atomic_tmp.replace(TXT_OCCURRENCE_INVENTORY)
inventory.to_csv(REPORT_ROOT/'txt_occurrence_inventory.csv',index=False)
valid=inventory.loc[inventory.hash_status.eq('PASS')&inventory.txt_sha256.notna()].copy()
unique_inventory=valid.loc[valid.occurrence_order.eq(0)].copy().reset_index(drop=True)
unique_inventory['physical_occurrence_count']=unique_inventory.txt_sha256.map(valid.txt_sha256.value_counts())
tmp=TXT_UNIQUE_INVENTORY.with_suffix('.tmp.parquet');unique_inventory.to_parquet(tmp,index=False,compression='zstd');tmp.replace(TXT_UNIQUE_INVENTORY)
print({'physical_paths':len(inventory),'hash_pass':int(inventory.hash_status.eq('PASS').sum()),'hash_fail':int(inventory.hash_status.eq('FAIL').sum()),'unique_hashes_available':int(valid.txt_sha256.nunique()),'duplicate_paths':int(len(valid)-valid.txt_sha256.nunique())})


{'physical_txt_paths': 108174, 'hashes_reused_from_checkpoint': 0, 'hashes_pending': 108174, 'hash_workers': 32}
Hashed/checkpointed 1,000/108,174
Hashed/checkpointed 2,000/108,174
Hashed/checkpointed 3,000/108,174
Hashed/checkpointed 4,000/108,174
Hashed/checkpointed 5,000/108,174
Hashed/checkpointed 6,000/108,174
Hashed/checkpointed 7,000/108,174
Hashed/checkpointed 8,000/108,174
Hashed/checkpointed 9,000/108,174
Hashed/checkpointed 10,000/108,174
Hashed/checkpointed 11,000/108,174
Hashed/checkpointed 12,000/108,174
Hashed/checkpointed 13,000/108,174
Hashed/checkpointed 14,000/108,174
Hashed/checkpointed 15,000/108,174
Hashed/checkpointed 16,000/108,174
Hashed/checkpointed 17,000/108,174
Hashed/checkpointed 18,000/108,174
Hashed/checkpointed 19,000/108,174
Hashed/checkpointed 20,000/108,174
Hashed/checkpointed 21,000/108,174
Hashed/checkpointed 22,000/108,174
Hashed/checkpointed 23,000/108,174
Hashed/checkpointed 24,000/108,174
Hashed/checkpointed 25,000/108,174
Hashed/checkpointed 2

In [15]:
# CELL 04 - Consolidate aircraft identity evidence across all physical occurrences
UUID_RE = re.compile(r'^[0-9A-F]{8}-[0-9A-F]{4}-[0-9A-F]{4}-[0-9A-F]{4}-[0-9A-F]{12}$', re.I)
US_REG_RE = re.compile(r'^N[0-9A-Z]{1,5}$', re.I)
HYPHEN_REG_RE = re.compile(r'^[A-Z0-9]{1,3}-[A-Z0-9]{2,5}$', re.I)

def aircraft_candidate_from_filename(path):
    stem = Path(path).stem.upper()
    tokens = [x.strip(' _.,;()[]{}') for x in re.split(r'[_\s.]+', stem) if x.strip()]
    candidates=[]
    for pos, token in enumerate(tokens):
        if UUID_RE.fullmatch(token) or token.isdigit():
            continue
        if US_REG_RE.fullmatch(token):
            candidates.append((0,pos,token,'SOURCE_FILENAME_US_REGISTRATION'))
        elif HYPHEN_REG_RE.fullmatch(token):
            candidates.append((1,pos,token,'SOURCE_FILENAME_HYPHENATED_REGISTRATION'))
    return sorted(candidates)[0][2:] if candidates else (None,None)

identity_rows=[]
for row in valid[['txt_sha256','txt_file']].itertuples(index=False):
    value, source = aircraft_candidate_from_filename(row.txt_file)
    identity_rows.append({'txt_sha256':row.txt_sha256,'txt_file':row.txt_file,
                          'aircraft_identifier_candidate':value,
                          'aircraft_identifier_candidate_source':source})
identity_occurrences=pd.DataFrame(identity_rows)

def consolidate_identity(group):
    values=sorted(set(group.aircraft_identifier_candidate.dropna().astype(str).str.upper()))
    sources=sorted(set(group.aircraft_identifier_candidate_source.dropna().astype(str)))
    if len(values)==1:
        status='RESOLVED_CONSISTENT_OCCURRENCES'; value=values[0]
    elif len(values)>1:
        status='CONFLICT_ACROSS_IDENTICAL_CONTENT_OCCURRENCES'; value=None
    else:
        status='UNRESOLVED_FROM_FILENAME'; value=None
    return pd.Series({'aircraft_identifier_occurrence':value,
                      'aircraft_identity_occurrence_status':status,
                      'aircraft_identity_occurrence_values_json':json.dumps(values),
                      'aircraft_identity_occurrence_sources_json':json.dumps(sources)})

identity_by_hash=(identity_occurrences.groupby('txt_sha256',as_index=False)
                  .apply(consolidate_identity, include_groups=False).reset_index())
if 'level_0' in identity_by_hash.columns: identity_by_hash=identity_by_hash.drop(columns='level_0')
if 'index' in identity_by_hash.columns and 'txt_sha256' not in identity_by_hash.columns:
    identity_by_hash=identity_by_hash.rename(columns={'index':'txt_sha256'})
# pandas groupby.apply shape guard
if 'txt_sha256' not in identity_by_hash.columns:
    identity_by_hash = pd.concat([
        group.assign(txt_sha256=sha) for sha, group in []
    ], ignore_index=True)
# Rebuild deterministically if apply did not preserve the key
if len(identity_by_hash) != valid.txt_sha256.nunique() or 'txt_sha256' not in identity_by_hash.columns:
    out=[]
    for sha,g in identity_occurrences.groupby('txt_sha256'):
        r=consolidate_identity(g).to_dict();r['txt_sha256']=sha;out.append(r)
    identity_by_hash=pd.DataFrame(out)
identity_by_hash.to_parquet(INVENTORY_ROOT/'aircraft_identity_by_hash.parquet',index=False,compression='zstd')
print(identity_by_hash.aircraft_identity_occurrence_status.value_counts(dropna=False))


aircraft_identity_occurrence_status
RESOLVED_CONSISTENT_OCCURRENCES    61903
UNRESOLVED_FROM_FILENAME           46065
Name: count, dtype: int64


In [16]:
# CELL 05 - Row-oriented TXT parser
KNOWN_SECTIONS={'LUBE_OFBYP','ECU_STATISTICS','ENGINE_STATISTICS','CHIP','STARTS','TKO_PERFORMANCE','CLM_PERFORMANCE','CRU_PERFORMANCE','ENGINE_REDLINE','INCIDENT_RECORD_A','USAGE_STATISTICS','LANDING_STATISTICS','IGNITORA_LIFE_USAGE','IGNITORB_LIFE_USAGE','MC_ATTRIBUTES','MC_HISTORY_ATTRIBUTES','FF_ATTRIBUTES','ECU_TIME_AT_TEMP','ENGINE_TIME_AT_PLA','N1_MAJOR_SPEED_CYCLES','N1_MINOR_SPEED_CYCLES','N2_MAJOR_SPEED_CYCLES','N2_MINOR_SPEED_CYCLES','ITT_MAJOR_TEMPERATURE_CYCLES','ITT_MINOR_TEMPERATURE_CYCLES','P3_MAJOR_PRESSURE_CYCLES','N1_DWELL_TIME','N2_DWELL_TIME','ITT_DWELL_TIME','N1_ITT_DWELL_CYCLES','N2_ITT_DWELL_CYCLES','N2_T3_DWELL_CYCLES','N1_TR_PEAK_SPEED_CYCLES','N2_TR_PEAK_SPEED_CYCLES','TR_UTILIZATION','INCIDENT_RECORD_B','TYPE1_THRESHOLD','IGNITION_RECORD','RECORD_FAILURES','TYPE0_THRESHOLD','P3_MINOR_CYCLES_1','P3_MINOR_CYCLES_2','P3_MINOR_CYCLES_3','P3_MINOR_CYCLES_4','P3_MINOR_CYCLES_5','P3_MINOR_CYCLES_6','P3_MINOR_CYCLES_7','P3_MINOR_CYCLES_8','P3_MINOR_CYCLES_9'}

def clean_section(v): return re.sub(r'[^A-Z0-9]+','_',str(v).upper()).strip('_')
def split_tab(line): return [x.strip() for x in line.rstrip('\r\n').split('\t')]
def is_section(line):
    s=line.strip(); return '\t' not in s and clean_section(s) in KNOWN_SECTIONS
def is_header(fields):
    if len(fields)<2:return False
    alpha=sum(bool(re.search(r'[A-Za-z]',x)) for x in fields)
    numeric=sum(pd.notna(pd.to_numeric(x,errors='coerce')) for x in fields)
    return alpha>=max(2,len(fields)//2) and alpha>numeric
def normalize_key(v): return re.sub(r'[^a-z0-9]','',str(v).lower())
def find_value(payload,contains_all=None,aliases=None):
    for k,v in payload.items():
        nk=normalize_key(k)
        if aliases and nk in aliases:return v
        if contains_all and all(x in nk for x in contains_all):return v
    return None
def to_num(v):
    z=pd.to_numeric(v,errors='coerce'); return float(z) if pd.notna(z) else np.nan
def to_word(v):
    z=to_num(v); return None if pd.isna(z) else int(z)&0xffffffff
def clean_six_digits(value):
    if value is None:return None
    try:
        if pd.isna(value):return None
    except Exception:pass
    text=re.sub(r'\D','',str(value).strip().split('.',1)[0])
    return text.zfill(6)[-6:] if text else None

def parse_compact_gmt(date_value,time_value):
    date_text=clean_six_digits(date_value);time_text=clean_six_digits(time_value)
    if date_text is None or time_text is None:return pd.NaT
    parsed=pd.to_datetime(date_text+time_text,format='%y%m%d%H%M%S',utc=True,errors='coerce')
    if pd.isna(parsed) or parsed<MIN_VALID_TIMESTAMP or parsed>MAX_VALID_TIMESTAMP:return pd.NaT
    return parsed

def section_timestamp_fields(section,payload):
    section_norm=clean_section(section)
    if section_norm=='FF_ATTRIBUTES':
        date_value=find_value(payload,aliases={'ffgmtdate'})
        time_value=find_value(payload,aliases={'ffgmttime'})
        source='FF GMT Date + FF GMT Time'
    elif section_norm=='LANDING_STATISTICS':
        date_value=find_value(payload,aliases={'landinggmtdate'})
        time_value=find_value(payload,aliases={'landinggmttime'})
        source='Landing GMT Date + Landing GMT Time'
    else:
        date_value=find_value(payload,contains_all=['gmt','date'])
        time_value=find_value(payload,contains_all=['gmt','time'])
        source='Generic section GMT fields' if date_value is not None or time_value is not None else 'No explicit GMT fields'
    parsed=parse_compact_gmt(date_value,time_value)
    quality='NO_EXPLICIT_GMT_FIELDS' if date_value is None and time_value is None else ('VALID_EXPLICIT_GMT' if pd.notna(parsed) else 'UNPARSEABLE_OR_OUT_OF_RANGE')
    return {
        'record_date_raw':None if date_value is None else str(date_value).strip(),
        'record_time_raw':None if time_value is None else str(time_value).strip(),
        'record_datetime_utc':parsed,
        'timestamp_source':source,
        'timestamp_format':'YYMMDD HHMMSS',
        'timestamp_quality':quality,
        'timestamp_contract_version':TIMESTAMP_CONTRACT_VERSION,
    }
def row_uid(sha,section,line,payload):
    return hashlib.sha256((sha+'|'+section+'|'+str(line)+'|'+json.dumps(payload,sort_keys=True,default=str)).encode()).hexdigest()

def parse_file(item):
    path=Path(item['txt_file']);sha=item['txt_sha256']
    status={'txt_file':str(path),'txt_sha256':sha,'status':'FAIL','stage':'PARSE','records':0,'fault_rows':0,'source_lines':0,'accounted_lines':0,'unclassified_lines':0,'error':''}
    try:
        lines=path.read_text(encoding='latin-1',errors='replace').splitlines();status['source_lines']=len(lines)
        output=[];section='UNSECTIONED';i=0;section_record=0;classified=set();file_engine=file_model=file_channel=None
        while i<len(lines):
            raw=lines[i];s=raw.strip();line_no=i+1
            if not s:i+=1;continue
            if is_section(raw):section=clean_section(s);section_record=0;classified.add(line_no);i+=1;continue
            fields=split_tab(raw)
            if is_header(fields):
                header=[x or f'_unnamed_{n+1}' for n,x in enumerate(fields)];classified.add(line_no);j=i+1
                while j<len(lines):
                    candidate=lines[j];cs=candidate.strip();candidate_no=j+1
                    if not cs:j+=1;continue
                    if is_section(candidate):break
                    values=split_tab(candidate)
                    if is_header(values):break
                    if len(values)==1 and len(header)>1 and not re.search(r'\d',values[0]):break
                    if len(values)<len(header):values += [None]*(len(header)-len(values))
                    extras=values[len(header):];values=values[:len(header)];payload=dict(zip(header,values))
                    for n,x in enumerate(extras,1):payload[f'_extra_value_{n}']=x
                    section_record+=1;classified.add(candidate_no)
                    engine=find_value(payload,aliases={'engineserialnumber','engineserial','engineesn','esn'})
                    model=find_value(payload,aliases={'enginemodelnumber','enginemodel','modelnumber','modelno'})
                    channel=find_value(payload,aliases={'ecuchannel','channel','ecunumber','ecuidlr'})
                    if engine:file_engine=str(engine).strip()
                    if model:file_model=str(model).strip().upper()
                    if channel:file_channel=str(channel).strip()
                    leg=find_value(payload,contains_all=['leg','number']) or find_value(payload,aliases={'leg','flightleg'})
                    ecu_time=find_value(payload,contains_all=['ecu','operating','time'])
                    timestamp_fields=section_timestamp_fields(section,payload)
                    sw={n:find_value(payload,contains_all=['engine','status','word',str(n)]) for n in (1,2,3,4)}
                    fault=find_value(payload,aliases={'fffaultid','faultid','faultcode','ffid'})
                    output.append({'record_uid':row_uid(sha,section,candidate_no,payload),'txt_sha256':sha,'source_txt_file':str(path),'source_line_start':candidate_no,'source_line_end':candidate_no,'section':section,'section_record_number':section_record,'payload_json':json.dumps(payload,ensure_ascii=False,separators=(',',':')),'engine_serial_observed':None if engine is None else str(engine).strip(),'engine_model_observed':None if model is None else str(model).strip().upper(),'ecu_channel_observed':None if channel is None else str(channel).strip(),'leg_number':to_num(leg),'ecu_operating_time_raw':to_num(ecu_time),**timestamp_fields,'engine_status_word_1_raw':to_word(sw[1]),'engine_status_word_2_raw':to_word(sw[2]),'engine_status_word_3_raw':to_word(sw[3]),'engine_status_word_4_raw':to_word(sw[4]),'fault_code':None if fault is None else str(fault).strip().replace('.0',''),'parser_version':PARSER_VERSION})
                    j+=1
                i=j;continue
            section_record+=1;classified.add(line_no);payload={'_raw_line':s}
            output.append({'record_uid':row_uid(sha,section,line_no,payload),'txt_sha256':sha,'source_txt_file':str(path),'source_line_start':line_no,'source_line_end':line_no,'section':section,'section_record_number':section_record,'payload_json':json.dumps(payload,separators=(',',':')),'leg_number':np.nan,'ecu_operating_time_raw':np.nan,'record_date_raw':None,'record_time_raw':None,'record_datetime_utc':pd.NaT,'timestamp_source':'No explicit GMT fields','timestamp_format':'YYMMDD HHMMSS','timestamp_quality':'NO_EXPLICIT_GMT_FIELDS','timestamp_contract_version':TIMESTAMP_CONTRACT_VERSION,'engine_status_word_1_raw':None,'engine_status_word_2_raw':None,'engine_status_word_3_raw':None,'engine_status_word_4_raw':None,'fault_code':None,'parser_version':PARSER_VERSION,'row_classification':'UNCLASSIFIED_RAW'})
            i+=1
        if not output:raise ValueError('No rows emitted')
        for r in output:
            r['engine_serial']=r.get('engine_serial_observed') or file_engine
            r['engine_model_current_record']=r.get('engine_model_observed') or file_model
            r['ecu_channel']=r.get('ecu_channel_observed') or file_channel
        status.update({'status':'PASS','records':len(output),'fault_rows':sum(bool(r.get('fault_code')) for r in output),'engine_serial':file_engine,'engine_model':file_model,'ecu_channel':file_channel,'accounted_lines':len(classified),'unclassified_lines':sum(r.get('row_classification')=='UNCLASSIFIED_RAW' for r in output)})
        return status,output
    except Exception:
        status['error']=traceback.format_exc(limit=4);return status,[]
print('Parser ready')


Parser ready


In [17]:
# CELL 06 - Complete model-aware SW1-SW4 decoder and stable output schema
REQ=['StatusWord','Bit_Field_Description','LSB','MSB','Type','ModelNo','Value_Bit','Value_Description']
engine='xlrd' if DECODER_XLSX.suffix.lower()=='.xls' else 'openpyxl'
decoder=pd.read_excel(DECODER_XLSX,engine=engine);decoder.columns=[str(c).strip() for c in decoder.columns]
missing=set(REQ)-set(decoder.columns)
if missing:raise ValueError(f'Decoder columns missing: {sorted(missing)}')
decoder=decoder[REQ].copy();decoder['ModelNo']=decoder.ModelNo.astype('string').str.strip().str.upper()
for c in ['StatusWord','LSB','MSB']:decoder[c]=pd.to_numeric(decoder[c],errors='coerce').astype('Int64')
decoder=decoder.loc[decoder.ModelNo.notna() & decoder.StatusWord.isin([1,2,3,4]) & decoder.LSB.notna() & decoder.MSB.notna()].copy()

def value_int(v):
    if pd.isna(v):return None
    t=re.sub(r'\s+','',str(v))
    if re.fullmatch(r'0[bB][01]+',t):return int(t[2:],2)
    if re.fullmatch(r'[01]+',t):return int(t,2)
    try:return int(float(t))
    except:return None
def bits(w,l,h):
    if w is None or pd.isna(w):return None
    return (int(w)>>int(l))&((1<<(int(h)-int(l)+1))-1)
def slug(v):
    s=str(v).lower();s=re.sub(r'\bthrust\s+reverser\b','tr',s);s=re.sub(r'\bweight\s+on\s+wheels\b','wow',s)
    return re.sub(r'_+','_',re.sub(r'[^a-z0-9]+','_',s)).strip('_') or 'unnamed_field'

decoder['Value_Int']=decoder.Value_Bit.map(value_int);decoder['field_slug']=decoder.Bit_Field_Description.map(slug)
COMMON={1:{'sw1_wow':(22,22),'sw1_wheel_spin_up':(24,24)},3:{'sw3_tr_unlock_1plu':(0,0),'sw3_tr_stowed':(1,1),'sw3_tr_deployed':(2,2),'sw3_tr_position_indeterminate':(3,3),'sw3_tr_unlock_2plu':(4,4),'sw3_tr_deploy_command':(5,5),'sw3_tr_stow_command':(6,6),'sw3_tr_isolation_solenoid_command':(9,9),'sw3_tr_dcu_solenoid_command':(10,10),'sw3_tr_isolation_pressure_switch':(12,12),'sw3_tr_hydraulic_control_1':(18,18),'sw3_tr_hydraulic_control_2':(19,19),'sw3_tr_position_resistor_fault':(20,20),'sw3_tr_state_error':(21,21),'sw3_tr_solenoid_fault':(22,22),'sw3_tr_controller_state_code':(23,25),'sw3_tr_unsafe':(26,26)}}
BY_LOC={(sw,l,h):n for sw,d in COMMON.items() for n,(l,h) in d.items()}
fields=decoder[['ModelNo','StatusWord','LSB','MSB','Bit_Field_Description','Type','field_slug']].drop_duplicates().copy()
fields['output_column']=fields.apply(lambda r:BY_LOC.get((int(r.StatusWord),int(r.LSB),int(r.MSB)),f"sw{int(r.StatusWord)}_{r.field_slug}"),axis=1)
# avoid same-model normalized-name collisions
cnt=fields.groupby(['ModelNo','output_column']).output_column.transform('size')
fields.loc[cnt.gt(1),'output_column']=fields.loc[cnt.gt(1)].apply(lambda r:f"{r.output_column}__b{int(r.LSB):02d}_{int(r.MSB):02d}",axis=1)
ALL_DECODED_COLUMNS=sorted(set(fields.output_column)|{n for d in COMMON.values() for n in d})
MODEL_FIELDS={m:g.to_dict('records') for m,g in fields.groupby('ModelNo')};supported_models=set(MODEL_FIELDS)
LOOKUP={}
for key,g in decoder.loc[decoder.Value_Int.notna()].groupby(['ModelNo','StatusWord','LSB','MSB','Bit_Field_Description']):
    LOOKUP[key]={int(r.Value_Int):str(r.Value_Description) for r in g.itertuples() if pd.notna(r.Value_Description)}
DECODER_SHA256=hashlib.sha256(DECODER_XLSX.read_bytes()).hexdigest()

identity_map=identity_by_hash.set_index('txt_sha256').to_dict('index')
def decode_rows(rows,model,model_status):
    model=str(model).strip().upper() if model else ''
    out=[]
    for r in rows:
        result={c:pd.NA for c in ALL_DECODED_COLUMNS}; words={n:r.get(f'engine_status_word_{n}_raw') for n in (1,2,3,4)}
        for sw,d in COMMON.items():
            for name,(l,h) in d.items():result[name]=bits(words.get(sw),l,h)
        details=[];unmapped=0;decoded_count=0
        for f in MODEL_FIELDS.get(model,[]):
            sw,l,h=int(f['StatusWord']),int(f['LSB']),int(f['MSB']);raw=bits(words.get(sw),l,h);col=f['output_column']
            if raw is not None:result[col]=raw;decoded_count+=1
            desc=LOOKUP.get((model,sw,l,h,f['Bit_Field_Description']),{}).get(raw) if raw is not None else None
            if raw is not None and desc is None:unmapped+=1
            details.append({'status_word':sw,'lsb':l,'msb':h,'field_name':str(f['Bit_Field_Description']),'output_column':col,'raw_value':raw,'value_description':desc})
        st,dp,ind=result.get('sw3_tr_stowed'),result.get('sw3_tr_deployed'),result.get('sw3_tr_position_indeterminate')
        state='STOWED' if st==1 and dp==0 else 'DEPLOYED' if st==0 and dp==1 else 'TRANSITION' if st==0 and dp==0 else 'CONTRADICTORY' if st==1 and dp==1 else None
        ident=identity_map.get(r['txt_sha256'],{})
        result.update({'aircraft_identifier':ident.get('aircraft_identifier_occurrence',pd.NA),'aircraft_identifier_type':'REGISTRATION_OR_TAIL' if ident.get('aircraft_identifier_occurrence') else 'UNRESOLVED','aircraft_identifier_source':'PHYSICAL_OCCURRENCE_FILENAMES','aircraft_identity_status':ident.get('aircraft_identity_occurrence_status','UNRESOLVED'),'aircraft_registration':ident.get('aircraft_identifier_occurrence',pd.NA),'aircraft_serial_number':pd.NA,'engine_position':pd.NA,'cross_side_engine_serial':pd.NA,'cross_side_engine_resolution_status':'PENDING_POST_PARSE_PAIRING','resolved_engine_model':model or pd.NA,'model_resolution_status':model_status,'decoder_status':'DECODED_MODEL_SPECIFIC_SW1_SW4' if model in supported_models else 'DECODED_COMMON_FIELDS_MODEL_UNAVAILABLE','decoder_version':DECODER_VERSION,'decoder_definition_sha256':DECODER_SHA256,'decoder_model_fields_available':len(MODEL_FIELDS.get(model,[])),'decoder_model_fields_decoded':decoded_count,'decoder_unmapped_value_count':unmapped,'decoded_status_fields_json':json.dumps(details,separators=(',',':')),'tr_physical_state':state,'tr_position_quality':'POSITION_INDETERMINATE_ACTIVE' if ind==1 else 'POSITION_BITS_CONTRADICTORY' if state=='CONTRADICTORY' else 'POSITION_NOT_OBSERVED' if state is None else 'POSITION_INTERPRETABLE'})
        out.append({**r,**result})
    return out

def atomic_parquet(df,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix('.tmp.parquet');df.to_parquet(tmp,index=False,compression='zstd');tmp.replace(path)

fields.to_csv(REPORT_ROOT/'decoder_field_catalog.csv',index=False)
print({'decoder_rows':len(decoder),'supported_models':len(supported_models),'flattened_columns':len(ALL_DECODED_COLUMNS),'definition_sha256':DECODER_SHA256})


{'decoder_rows': 1252, 'supported_models': 6, 'flattened_columns': 79, 'definition_sha256': '3da8459f0d47a4a6aa1aedbcf97cd9b5823f292eb661ba5fb268d9d5e229f370'}


In [18]:
# CELL 07 - Restartable parse, decode and write of every unique TXT hash
representatives=unique_inventory[['txt_file','txt_sha256']].copy()
if MAX_UNIQUE_FILES is not None:representatives=representatives.head(MAX_UNIQUE_FILES)
work_items=representatives.to_dict('records')
def output_partition(sha):return RECORD_ROOT/f'file_hash={sha}'/'part-00000.parquet'
prior=pd.read_parquet(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame()
pass_hashes=set(prior.loc[prior.status.eq('PASS'),'txt_sha256'].astype(str)) if not prior.empty and {'status','txt_sha256'}.issubset(prior.columns) else set()
completed={sha for sha in pass_hashes if output_partition(sha).is_file() and output_partition(sha).stat().st_size>0}
pending=[item for item in work_items if item['txt_sha256'] not in completed]
print({'total_selected':len(work_items),'already_passed_with_partition':len(completed),'pending':len(pending),'workers':MAX_WORKERS,'mode':'RESUME_FULL_REBUILD' if completed else 'FULL_REBUILD_FROM_ZERO'})

def process_one(item):
    status,rows=parse_file(item)
    if status.get('status')!='PASS':return status
    try:
        model=status.get('engine_model');decoded=decode_rows(rows,model,'MODEL_FROM_FILE' if model else 'MODEL_NOT_FOUND');atomic_parquet(pd.DataFrame(decoded),output_partition(item['txt_sha256']))
        status.update({'status':'PASS','stage':'DECODED_AND_WRITTEN','decoded_rows':len(decoded),'resolved_engine_model':model})
    except Exception:status.update({'status':'FAIL','stage':'DECODE_OR_WRITE','decoded_rows':0,'error':traceback.format_exc(limit=4)})
    return status

def commit(new_rows):
    existing=pd.read_parquet(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame()
    combined=pd.concat([existing,pd.DataFrame(new_rows)],ignore_index=True,sort=False) if not existing.empty else pd.DataFrame(new_rows)
    if not combined.empty:
        combined=combined.drop_duplicates('txt_sha256',keep='last');atomic_parquet(combined,MANIFEST_PATH)

for batch_no,start in enumerate(range(0,len(pending),BATCH_SIZE),1):
    batch=pending[start:start+BATCH_SIZE];batch_results=[]
    print(f'\n=== RESUMABLE BATCH {batch_no:04d} | {start+1}-{start+len(batch)} of {len(pending)} pending ===')
    with ProcessPoolExecutor(max_workers=min(MAX_WORKERS,len(batch)),mp_context=mp.get_context('fork')) as pool:
        futures={pool.submit(process_one,item):item for item in batch}
        for num,future in enumerate(as_completed(futures),1):
            item=futures[future]
            try:r=future.result()
            except Exception as exc:r={**item,'status':'FAIL','stage':'WORKER_EXCEPTION','error':repr(exc),'records':0,'decoded_rows':0}
            batch_results.append(r)
            if num==1 or num%PRINT_EVERY==0 or num==len(batch):print(f'[{num:,}/{len(batch):,}] PASS={sum(x.get("status")=="PASS" for x in batch_results)} FAIL={sum(x.get("status")!="PASS" for x in batch_results)}')
    commit(batch_results);print(f'=== BATCH {batch_no:04d} COMMITTED ===')
manifest=pd.read_parquet(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame()
print(manifest.status.value_counts(dropna=False) if not manifest.empty else 'No manifest rows')
print('Partitions written:',sum(1 for _ in RECORD_ROOT.glob('file_hash=*/part-*.parquet')))


{'total_selected': 107968, 'already_passed_with_partition': 0, 'pending': 107968, 'workers': 60, 'mode': 'FULL_REBUILD_FROM_ZERO'}

=== RESUMABLE BATCH 0001 | 1-500 of 107968 pending ===


[1/500] PASS=1 FAIL=0
[100/500] PASS=100 FAIL=0
[200/500] PASS=200 FAIL=0
[300/500] PASS=300 FAIL=0
[400/500] PASS=400 FAIL=0
[500/500] PASS=500 FAIL=0
=== BATCH 0001 COMMITTED ===

=== RESUMABLE BATCH 0002 | 501-1000 of 107968 pending ===
[1/500] PASS=1 FAIL=0
[100/500] PASS=100 FAIL=0
[200/500] PASS=200 FAIL=0
[300/500] PASS=300 FAIL=0
[400/500] PASS=400 FAIL=0
[500/500] PASS=500 FAIL=0
=== BATCH 0002 COMMITTED ===

=== RESUMABLE BATCH 0003 | 1001-1500 of 107968 pending ===
[1/500] PASS=1 FAIL=0
[100/500] PASS=100 FAIL=0
[200/500] PASS=200 FAIL=0
[300/500] PASS=300 FAIL=0
[400/500] PASS=400 FAIL=0
[500/500] PASS=500 FAIL=0
=== BATCH 0003 COMMITTED ===

=== RESUMABLE BATCH 0004 | 1501-2000 of 107968 pending ===
[1/500] PASS=1 FAIL=0
[100/500] PASS=100 FAIL=0
[200/500] PASS=200 FAIL=0
[300/500] PASS=300 FAIL=0
[400/500] PASS=400 FAIL=0
[500/500] PASS=500 FAIL=0
=== BATCH 0004 COMMITTED ===

=== RESUMABLE BATCH 0005 | 2001-2500 of 107968 pending ===
[1/500] PASS=1 FAIL=0
[100/500] PASS=

In [19]:
# =============================================================================
# CELL 08 - Restartable cross-side engine pairing, corrected
# Run after the parse/decode/write cell has completed the selected population.
# =============================================================================

from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from pathlib import Path
import traceback

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# -----------------------------------------------------------------------------
# 08.01 Build the pairing inventory from ALL currently available partitions.
# pair_pending is intentionally NOT used here. It is defined only after the
# fleet pairing model has been rebuilt from the complete current population.
# -----------------------------------------------------------------------------
parts = sorted(
    RECORD_ROOT.glob("file_hash=*/part-*.parquet")
)

if not parts:
    raise RuntimeError(
        f"No ROW V2 partitions found under {RECORD_ROOT}"
    )

minimal = []
read_failures = []

for idx, path in enumerate(parts, start=1):
    try:
        schema = set(
            pq.ParquetFile(path).schema_arrow.names
        )

        columns = [
            column
            for column in [
                "txt_sha256",
                "aircraft_identifier",
                "engine_serial",
            ]
            if column in schema
        ]

        frame = pd.read_parquet(
            path,
            columns=columns,
        )

        for column in [
            "txt_sha256",
            "aircraft_identifier",
            "engine_serial",
        ]:
            if column not in frame.columns:
                frame[column] = pd.NA

        minimal.append(
            frame[
                [
                    "txt_sha256",
                    "aircraft_identifier",
                    "engine_serial",
                ]
            ].drop_duplicates()
        )

    except Exception as exc:
        read_failures.append(
            {
                "partition": str(path),
                "error": repr(exc),
            }
        )

    if (
        idx == 1
        or idx % 5000 == 0
        or idx == len(parts)
    ):
        print(
            f"Pairing inventory "
            f"{idx:,}/{len(parts):,}"
        )

if not minimal:
    raise RuntimeError(
        "No pairing inventory was built."
    )

pair_source = (
    pd.concat(
        minimal,
        ignore_index=True,
        sort=False,
    )
    .dropna(
        subset=[
            "aircraft_identifier",
            "engine_serial",
        ]
    )
    .drop_duplicates()
)

pair_source["aircraft_identifier"] = (
    pair_source["aircraft_identifier"]
    .astype(str)
    .str.strip()
    .str.upper()
)

pair_source["engine_serial"] = (
    pair_source["engine_serial"]
    .astype(str)
    .str.strip()
    .str.upper()
)

pair_source = pair_source.loc[
    pair_source["aircraft_identifier"].ne("")
    & pair_source["engine_serial"].ne("")
].copy()


aircraft_engines = (
    pair_source.groupby(
        "aircraft_identifier"
    )["engine_serial"]
    .agg(
        lambda values:
            sorted(set(values))
    )
    .to_dict()
)

exact_two = {
    aircraft: engines
    for aircraft, engines
    in aircraft_engines.items()
    if len(engines) == 2
}

ambiguous = {
    aircraft: engines
    for aircraft, engines
    in aircraft_engines.items()
    if len(engines) > 2
}

single_engine = {
    aircraft: engines
    for aircraft, engines
    in aircraft_engines.items()
    if len(engines) == 1
}

print(
    {
        "partitions": len(parts),
        "aircraft_with_exactly_two_engines":
            len(exact_two),
        "aircraft_with_one_engine_observed":
            len(single_engine),
        "aircraft_with_more_than_two_engines":
            len(ambiguous),
        "pairing_inventory_read_failures":
            len(read_failures),
    }
)

if read_failures:
    pd.DataFrame(
        read_failures
    ).to_csv(
        REPORT_ROOT
        / "pairing_inventory_read_failures.csv",
        index=False,
    )

# Persist the population used to make pairing decisions.
pair_population = pd.DataFrame(
    [
        {
            "aircraft_identifier": aircraft,
            "engine_count": len(engines),
            "engine_serials_json": json.dumps(engines),
            "pairing_population_status": (
                "EXACTLY_TWO_ENGINES"
                if len(engines) == 2
                else "ONE_ENGINE_OBSERVED"
                if len(engines) == 1
                else "MORE_THAN_TWO_HISTORICAL_ENGINES"
            ),
        }
        for aircraft, engines
        in aircraft_engines.items()
    ]
)

atomic_parquet(
    pair_population,
    CHECKPOINT_ROOT
    / "aircraft_engine_pairing_population.parquet",
)

# -----------------------------------------------------------------------------
# 08.02 Determine which partitions still require enrichment.
# A prior PASS is reusable only when the partition still exists and is nonempty.
# -----------------------------------------------------------------------------
prior_pair = (
    pd.read_parquet(
        PAIRING_MANIFEST_PATH
    )
    if PAIRING_MANIFEST_PATH.exists()
    else pd.DataFrame()
)

if (
    not prior_pair.empty
    and {
        "partition",
        "status",
    }.issubset(prior_pair.columns)
):
    reusable_pairing = prior_pair.loc[
        prior_pair["status"].eq("PASS")
    ].copy()

    pair_done = {
        str(path)
        for path
        in reusable_pairing["partition"]
        if Path(path).is_file()
        and Path(path).stat().st_size > 0
    }
else:
    pair_done = set()

pair_pending = [
    path
    for path in parts
    if str(path) not in pair_done
]

print(
    {
        "pairing_total": len(parts),
        "pairing_completed": len(pair_done),
        "pairing_pending": len(pair_pending),
    }
)

# -----------------------------------------------------------------------------
# 08.03 Enrich one partition.
# Status is assigned row-by-row, not once for the complete partition.
# -----------------------------------------------------------------------------
def enrich_pairing_partition(path):
    path = Path(path)

    try:
        frame = pd.read_parquet(path)

        if "aircraft_identifier" not in frame.columns:
            frame["aircraft_identifier"] = pd.NA

        if "engine_serial" not in frame.columns:
            frame["engine_serial"] = pd.NA

        aircraft_normalized = (
            frame["aircraft_identifier"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        engine_normalized = (
            frame["engine_serial"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        cross_side = []
        resolution_status = []

        for aircraft, engine in zip(
            aircraft_normalized,
            engine_normalized,
        ):
            if pd.isna(aircraft) or not aircraft:
                cross_side.append(pd.NA)
                resolution_status.append(
                    "UNRESOLVED_AIRCRAFT_IDENTITY"
                )
                continue

            engines = aircraft_engines.get(
                str(aircraft),
                [],
            )

            if len(engines) == 2:
                if pd.isna(engine) or not engine:
                    cross_side.append(pd.NA)
                    resolution_status.append(
                        "UNRESOLVED_ENGINE_SERIAL"
                    )
                elif str(engine) == engines[0]:
                    cross_side.append(engines[1])
                    resolution_status.append(
                        "RESOLVED_EXACT_TWO_ENGINE_POPULATION"
                    )
                elif str(engine) == engines[1]:
                    cross_side.append(engines[0])
                    resolution_status.append(
                        "RESOLVED_EXACT_TWO_ENGINE_POPULATION"
                    )
                else:
                    cross_side.append(pd.NA)
                    resolution_status.append(
                        "ENGINE_NOT_IN_AIRCRAFT_PAIRING_POPULATION"
                    )

            elif len(engines) > 2:
                cross_side.append(pd.NA)
                resolution_status.append(
                    "AMBIGUOUS_MORE_THAN_TWO_HISTORICAL_ENGINES"
                )

            elif len(engines) == 1:
                cross_side.append(pd.NA)
                resolution_status.append(
                    "PAIR_NOT_OBSERVED_ONE_ENGINE_ONLY"
                )

            else:
                cross_side.append(pd.NA)
                resolution_status.append(
                    "UNRESOLVED_NO_PAIRING_EVIDENCE"
                )

        frame["cross_side_engine_serial"] = (
            pd.Series(
                cross_side,
                index=frame.index,
                dtype="string",
            )
        )

        frame[
            "cross_side_engine_resolution_status"
        ] = pd.Series(
            resolution_status,
            index=frame.index,
            dtype="string",
        )

        atomic_parquet(
            frame,
            path,
        )

        return {
            "partition": str(path),
            "status": "PASS",
            "rows": len(frame),
            "paired_rows": int(
                frame[
                    "cross_side_engine_serial"
                ].notna().sum()
            ),
            "error": "",
        }

    except Exception:
        return {
            "partition": str(path),
            "status": "FAIL",
            "rows": 0,
            "paired_rows": 0,
            "error": traceback.format_exc(
                limit=4
            ),
        }

# -----------------------------------------------------------------------------
# 08.04 Restartable parallel enrichment and checkpoint publishing.
# -----------------------------------------------------------------------------
pair_results = []

if pair_pending:
    with ProcessPoolExecutor(
        max_workers=min(
            MAX_WORKERS,
            len(pair_pending),
        ),
        mp_context=mp.get_context("fork"),
    ) as pool:
        futures = {
            pool.submit(
                enrich_pairing_partition,
                path,
            ): path
            for path in pair_pending
        }

        for idx, future in enumerate(
            as_completed(futures),
            start=1,
        ):
            path = futures[future]

            try:
                result = future.result()
            except Exception:
                result = {
                    "partition": str(path),
                    "status": "FAIL",
                    "rows": 0,
                    "paired_rows": 0,
                    "error": traceback.format_exc(
                        limit=4
                    ),
                }

            pair_results.append(result)

            if (
                idx == 1
                or idx % 250 == 0
                or idx == len(pair_pending)
            ):
                print(
                    f"Pair-enriched "
                    f"{idx:,}/"
                    f"{len(pair_pending):,}"
                )

            # Checkpoint during the run, not only at the end.
            if (
                idx % 250 == 0
                or idx == len(pair_pending)
            ):
                new_pair = pd.DataFrame(
                    pair_results
                )

                pair_manifest = (
                    pd.concat(
                        [
                            prior_pair,
                            new_pair,
                        ],
                        ignore_index=True,
                        sort=False,
                    )
                    if not prior_pair.empty
                    else new_pair
                )

                pair_manifest = (
                    pair_manifest
                    .drop_duplicates(
                        "partition",
                        keep="last",
                    )
                )

                atomic_parquet(
                    pair_manifest,
                    PAIRING_MANIFEST_PATH,
                )

pair_manifest = (
    pd.read_parquet(
        PAIRING_MANIFEST_PATH
    )
    if PAIRING_MANIFEST_PATH.exists()
    else prior_pair
)

if pair_manifest.empty:
    raise RuntimeError(
        "No aircraft pairing manifest was produced."
    )

print("\nPAIRING STATUS")
print(
    pair_manifest["status"]
    .value_counts(dropna=False)
    .to_string()
)

print(
    {
        "pairing_manifest_rows":
            len(pair_manifest),
        "pairing_pass":
            int(
                pair_manifest[
                    "status"
                ].eq("PASS").sum()
            ),
        "pairing_fail":
            int(
                pair_manifest[
                    "status"
                ].eq("FAIL").sum()
            ),
        "paired_rows":
            int(
                pd.to_numeric(
                    pair_manifest[
                        "paired_rows"
                    ],
                    errors="coerce",
                ).fillna(0).sum()
            ),
    }
)

if pair_manifest["status"].eq("FAIL").any():
    print(
        "REVIEW: Some partitions failed pairing enrichment. "
        "Rerun Cell 08 in resume mode to retry only failures."
    )
else:
    print(
        "PASS: Restartable cross-side engine pairing completed."
    )


Pairing inventory 1/107,966
Pairing inventory 5,000/107,966
Pairing inventory 10,000/107,966
Pairing inventory 15,000/107,966
Pairing inventory 20,000/107,966
Pairing inventory 25,000/107,966
Pairing inventory 30,000/107,966
Pairing inventory 35,000/107,966
Pairing inventory 40,000/107,966
Pairing inventory 45,000/107,966
Pairing inventory 50,000/107,966
Pairing inventory 55,000/107,966
Pairing inventory 60,000/107,966
Pairing inventory 65,000/107,966
Pairing inventory 70,000/107,966
Pairing inventory 75,000/107,966
Pairing inventory 80,000/107,966
Pairing inventory 85,000/107,966
Pairing inventory 90,000/107,966
Pairing inventory 95,000/107,966
Pairing inventory 100,000/107,966
Pairing inventory 105,000/107,966
Pairing inventory 107,966/107,966
{'partitions': 107966, 'aircraft_with_exactly_two_engines': 440, 'aircraft_with_one_engine_observed': 0, 'aircraft_with_more_than_two_engines': 304, 'pairing_inventory_read_failures': 0}
{'pairing_total': 107966, 'pairing_completed': 0, 'pairin

In [20]:
# =============================================================================
# CELL 09 - Pilot-aware/full-population reconciliation and downstream contract
# =============================================================================

import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 09.01 Load persisted manifests
# -----------------------------------------------------------------------------

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Pipeline manifest not found: {MANIFEST_PATH}"
    )


if not PAIRING_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Pairing manifest not found: "
        f"{PAIRING_MANIFEST_PATH}"
    )


manifest = pd.read_parquet(
    MANIFEST_PATH
)

pair_manifest = pd.read_parquet(
    PAIRING_MANIFEST_PATH
)


required_manifest_columns = {
    "txt_sha256",
    "status",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(manifest.columns)
)


if missing_manifest_columns:
    raise ValueError(
        "Pipeline manifest is missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )


required_pair_columns = {
    "partition",
    "status",
}

missing_pair_columns = (
    required_pair_columns
    - set(pair_manifest.columns)
)


if missing_pair_columns:
    raise ValueError(
        "Pairing manifest is missing columns: "
        f"{sorted(missing_pair_columns)}"
    )


# -----------------------------------------------------------------------------
# 09.02 Establish total and expected populations
# -----------------------------------------------------------------------------

all_source_hashes = set(
    unique_inventory[
        "txt_sha256"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


selected_hashes = set(
    representatives[
        "txt_sha256"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


expected_hashes = (
    selected_hashes
    if PILOT_MODE
    else all_source_hashes
)


if not expected_hashes:
    raise RuntimeError(
        "No hashes are expected for "
        "the current run scope."
    )


physical_paths = sorted(
    RECORD_ROOT.glob(
        "file_hash=*/part-*.parquet"
    )
)


physical_hashes = {
    path.parent.name.replace(
        "file_hash=",
        "",
        1,
    )
    for path in physical_paths
}


manifest_status = (
    manifest["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


manifest_hashes = (
    manifest["txt_sha256"]
    .fillna("")
    .astype(str)
    .str.strip()
)


pass_hashes = set(
    manifest_hashes.loc[
        manifest_status.eq("PASS")
    ]
) - {""}


fail_hashes = set(
    manifest_hashes.loc[
        manifest_status.eq("FAIL")
    ]
) - {""}


missing_expected_partitions = (
    expected_hashes
    - physical_hashes
)


selected_without_manifest_pass = (
    expected_hashes
    - pass_hashes
)


selected_pipeline_failures = (
    expected_hashes
    & fail_hashes
)


unprocessed_source_hashes = (
    all_source_hashes
    - physical_hashes
)


unexpected_partitions = (
    physical_hashes
    - all_source_hashes
)


# -----------------------------------------------------------------------------
# 09.03 Reconcile pairing manifest against the active population
# -----------------------------------------------------------------------------

def partition_hash(value):

    try:
        return (
            Path(str(value))
            .parent
            .name
            .replace(
                "file_hash=",
                "",
                1,
            )
        )

    except Exception:
        return ""


pair_manifest = pair_manifest.copy()


pair_manifest[
    "txt_sha256_from_partition"
] = pair_manifest[
    "partition"
].map(
    partition_hash
)


pair_status = (
    pair_manifest["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


pair_pass_hashes = set(
    pair_manifest.loc[
        pair_status.eq("PASS"),
        "txt_sha256_from_partition",
    ].astype(str)
) - {""}


pair_fail_hashes = set(
    pair_manifest.loc[
        pair_status.eq("FAIL"),
        "txt_sha256_from_partition",
    ].astype(str)
) - {""}


selected_without_pairing_pass = (
    expected_hashes
    - pair_pass_hashes
)


selected_pairing_failures = (
    expected_hashes
    & pair_fail_hashes
)


# -----------------------------------------------------------------------------
# 09.04 Validate the canonical schema
#
# Pilot:
#     Validate every selected partition.
#
# Full population:
#     Validate a deterministic sample of up to 1,000 partitions.
#     The final validation cell performs additional population checks.
# -----------------------------------------------------------------------------

required_columns = {
    "record_uid",
    "txt_sha256",
    "source_txt_file",
    "source_line_start",
    "source_line_end",
    "section",
    "payload_json",
    "engine_serial",
    "record_date_raw",
    "record_time_raw",
    "record_datetime_utc",
    "timestamp_source",
    "timestamp_format",
    "timestamp_quality",
    "timestamp_contract_version",
    "engine_status_word_1_raw",
    "engine_status_word_2_raw",
    "engine_status_word_3_raw",
    "engine_status_word_4_raw",
    "decoder_status",
    "decoder_version",
    "decoder_definition_sha256",
    "decoded_status_fields_json",
    "aircraft_identifier",
    "aircraft_identity_status",
    "cross_side_engine_serial",
    "cross_side_engine_resolution_status",
}


selected_partition_paths = [
    path
    for path in physical_paths
    if (
        path.parent.name.replace(
            "file_hash=",
            "",
            1,
        )
        in expected_hashes
    )
]


schema_paths = (
    selected_partition_paths
    if PILOT_MODE
    else selected_partition_paths[:1000]
)


schema_rows = []


for index, path in enumerate(
    schema_paths,
    start=1,
):

    try:

        names = set(
            pq.ParquetFile(
                path
            )
            .schema_arrow
            .names
        )


        missing_columns = sorted(
            required_columns
            - names
        )


        schema_rows.append(
            {
                "partition":
                    str(path),

                "missing_columns":
                    json.dumps(
                        missing_columns
                    ),

                "status":
                    (
                        "PASS"
                        if not missing_columns
                        else "FAIL"
                    ),

                "error":
                    "",
            }
        )


    except Exception as exc:

        schema_rows.append(
            {
                "partition":
                    str(path),

                "missing_columns":
                    json.dumps([]),

                "status":
                    "READ_FAIL",

                "error":
                    repr(exc),
            }
        )


    if (
        index == 1
        or index % 250 == 0
        or index == len(schema_paths)
    ):
        print(
            f"Schema checked "
            f"{index:,}/"
            f"{len(schema_paths):,}"
        )


schema_audit = pd.DataFrame(
    schema_rows
)


if schema_audit.empty:

    schema_audit = pd.DataFrame(
        columns=[
            "partition",
            "missing_columns",
            "status",
            "error",
        ]
    )


schema_audit_path = (
    REPORT_ROOT
    / (
        "pilot_schema_audit.csv"
        if PILOT_MODE
        else "full_schema_sample_audit.csv"
    )
)


schema_audit.to_csv(
    schema_audit_path,
    index=False,
)


schema_failures = int(
    schema_audit[
        "status"
    ].ne(
        "PASS"
    ).sum()
)


# -----------------------------------------------------------------------------
# 09.05 Build a scope-aware summary
# -----------------------------------------------------------------------------

hash_failures = int(
    inventory[
        "hash_status"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .eq(
        "FAIL"
    )
    .sum()
)


summary = {
    "run_id":
        RUN_ID,

    "run_scope":
        (
            "PILOT"
            if PILOT_MODE
            else "FULL_POPULATION"
        ),

    "pilot_mode":
        bool(PILOT_MODE),

    "physical_txt_paths_rehashed":
        int(
            len(inventory)
        ),

    "hash_failures":
        hash_failures,

    "unique_txt_hashes_total":
        int(
            len(
                all_source_hashes
            )
        ),

    "unique_hashes_expected_this_run":
        int(
            len(
                expected_hashes
            )
        ),

    "duplicate_physical_paths":
        int(
            len(valid)
            - valid[
                "txt_sha256"
            ].nunique()
        ),

    "pipeline_pass_hashes_in_scope":
        int(
            len(
                pass_hashes
                & expected_hashes
            )
        ),

    "pipeline_fail_hashes_in_scope":
        int(
            len(
                selected_pipeline_failures
            )
        ),

    "physical_partitions_total":
        int(
            len(
                physical_hashes
            )
        ),

    "expected_partitions_present":
        int(
            len(
                physical_hashes
                & expected_hashes
            )
        ),

    "missing_expected_partitions":
        int(
            len(
                missing_expected_partitions
            )
        ),

    "unprocessed_source_hashes":
        int(
            len(
                unprocessed_source_hashes
            )
        ),

    "unexpected_partitions":
        int(
            len(
                unexpected_partitions
            )
        ),

    "selected_without_manifest_pass":
        int(
            len(
                selected_without_manifest_pass
            )
        ),

    "pairing_pass_hashes_in_scope":
        int(
            len(
                pair_pass_hashes
                & expected_hashes
            )
        ),

    "pairing_fail_hashes_in_scope":
        int(
            len(
                selected_pairing_failures
            )
        ),

    "selected_without_pairing_pass":
        int(
            len(
                selected_without_pairing_pass
            )
        ),

    "schema_partitions_checked":
        int(
            len(
                schema_audit
            )
        ),

    "schema_failures":
        schema_failures,

    "output_root":
        str(
            OUTPUT_ROOT
        ),

    "schema_audit":
        str(
            schema_audit_path
        ),
}


summary_path = (
    REPORT_ROOT
    / (
        "pilot_run_summary.json"
        if PILOT_MODE
        else "full_run_summary.json"
    )
)


summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nCELL 09 RECONCILIATION SUMMARY"
)


print(
    json.dumps(
        summary,
        indent=2,
    )
)


# -----------------------------------------------------------------------------
# 09.06 Scope-aware blocking checks
# -----------------------------------------------------------------------------

blocking_errors = []


if summary[
    "hash_failures"
] != 0:

    blocking_errors.append(
        "Some TXT files could not be hashed."
    )


if summary[
    "missing_expected_partitions"
] != 0:

    blocking_errors.append(
        "Some hashes expected for the current "
        "scope lack output partitions."
    )


if summary[
    "selected_without_manifest_pass"
] != 0:

    blocking_errors.append(
        "Some hashes expected for the current "
        "scope are not PASS in the pipeline manifest."
    )


if summary[
    "pipeline_fail_hashes_in_scope"
] != 0:

    blocking_errors.append(
        "The current scope contains "
        "pipeline failures."
    )


if summary[
    "unexpected_partitions"
] != 0:

    blocking_errors.append(
        "Output contains partitions absent "
        "from the refreshed TXT inventory."
    )


if summary[
    "selected_without_pairing_pass"
] != 0:

    blocking_errors.append(
        "Some hashes expected for the current "
        "scope are not PASS in the pairing manifest."
    )


if summary[
    "pairing_fail_hashes_in_scope"
] != 0:

    blocking_errors.append(
        "The current scope contains "
        "pairing-enrichment failures."
    )


if summary[
    "schema_failures"
] != 0:

    blocking_errors.append(
        "Some checked partitions failed "
        "the canonical schema contract."
    )


if blocking_errors:

    print(
        "\nBLOCKING ERRORS"
    )

    for error in blocking_errors:
        print(
            " -",
            error,
        )

    raise RuntimeError(
        "Cell 09 reconciliation failed. "
        "Review the summary and schema audit."
    )


# -----------------------------------------------------------------------------
# 09.07 Final pilot/full status
# -----------------------------------------------------------------------------

if PILOT_MODE:

    if (
        len(expected_hashes)
        != PILOT_UNIQUE_FILES
    ):

        raise RuntimeError(
            "Pilot selection count does not "
            "match PILOT_UNIQUE_FILES: "
            f"selected={len(expected_hashes):,}, "
            f"configured={PILOT_UNIQUE_FILES:,}"
        )


    print(
        "\nPASS: Pilot reconciliation completed."
    )


    print(
        f"{len(unprocessed_source_hashes):,} "
        "source hashes remain intentionally "
        "unprocessed because PILOT_MODE=True."
    )


    print(
        "\nEXPANSION SETTINGS"
    )

    print(
        "PILOT_MODE = False"
    )

    print(
        "INITIALIZE_FULL_REBUILD = False"
    )

    print(
        "RESUME_REBUILD = True"
    )


else:

    if unprocessed_source_hashes:

        raise RuntimeError(
            "Full-population mode is active, "
            "but source hashes remain unprocessed: "
            f"{len(unprocessed_source_hashes):,}"
        )


    if (
        len(physical_hashes)
        != len(all_source_hashes)
    ):

        raise RuntimeError(
            "Full-population partition count "
            "does not equal the refreshed unique "
            "TXT hash count."
        )


    print(
        "\nPASS: Full-population reconciliation completed."
    )


    print(
        "Outputs are ready for the "
        "final validation cell."
    )

Schema checked 1/1,000
Schema checked 250/1,000
Schema checked 500/1,000
Schema checked 750/1,000
Schema checked 1,000/1,000

CELL 09 RECONCILIATION SUMMARY
{
  "run_id": "11001_ROW_V2_TIMESTAMP_CORRECTED_20260920",
  "run_scope": "FULL_POPULATION",
  "pilot_mode": false,
  "physical_txt_paths_rehashed": 108174,
  "hash_failures": 0,
  "unique_txt_hashes_total": 107968,
  "unique_hashes_expected_this_run": 107968,
  "duplicate_physical_paths": 206,
  "pipeline_pass_hashes_in_scope": 107966,
  "pipeline_fail_hashes_in_scope": 2,
  "physical_partitions_total": 107966,
  "expected_partitions_present": 107966,
  "missing_expected_partitions": 2,
  "unprocessed_source_hashes": 2,
  "unexpected_partitions": 0,
  "selected_without_manifest_pass": 2,
  "pairing_pass_hashes_in_scope": 107966,
  "pairing_fail_hashes_in_scope": 0,
  "selected_without_pairing_pass": 2,
  "schema_partitions_checked": 1000,
  "schema_failures": 0,
  "output_root": "/raid3/e296408/All_ECFRs_working/ECFR_11001_ROW_V2"

RuntimeError: Cell 09 reconciliation failed. Review the summary and schema audit.

In [22]:
# =============================================================================
# CELL 09 - Full-population reconciliation with approved terminal exclusions
#
# Contract:
# - Every source hash must be either:
#     1. PASS with a physical partition, or
#     2. an explicitly approved terminal exclusion.
# - Approved exclusions are written to a permanent audit file.
# - Unexpected output partitions remain blocking.
# - Schema validation is performed on up to 1,000 deterministic partitions
#   for a full run, or every selected partition for a pilot.
# =============================================================================

import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 09.01 Load persisted manifests
# -----------------------------------------------------------------------------

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Pipeline manifest not found: {MANIFEST_PATH}"
    )

if not PAIRING_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Pairing manifest not found: {PAIRING_MANIFEST_PATH}"
    )


manifest = pd.read_parquet(
    MANIFEST_PATH
)

pair_manifest = pd.read_parquet(
    PAIRING_MANIFEST_PATH
)


required_manifest_columns = {
    "txt_sha256",
    "status",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(manifest.columns)
)

if missing_manifest_columns:
    raise ValueError(
        "Pipeline manifest is missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )


required_pair_columns = {
    "partition",
    "status",
}

missing_pair_columns = (
    required_pair_columns
    - set(pair_manifest.columns)
)

if missing_pair_columns:
    raise ValueError(
        "Pairing manifest is missing columns: "
        f"{sorted(missing_pair_columns)}"
    )


manifest = (
    manifest.copy()
    .drop_duplicates(
        "txt_sha256",
        keep="last",
    )
    .reset_index(
        drop=True
    )
)

manifest["txt_sha256"] = (
    manifest["txt_sha256"]
    .fillna("")
    .astype(str)
    .str.strip()
)

manifest["status_norm"] = (
    manifest["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


pair_manifest = (
    pair_manifest.copy()
    .drop_duplicates(
        "partition",
        keep="last",
    )
    .reset_index(
        drop=True
    )
)

pair_manifest["status_norm"] = (
    pair_manifest["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


# -----------------------------------------------------------------------------
# 09.02 Establish source, selected, and physical populations
# -----------------------------------------------------------------------------

all_source_hashes = set(
    unique_inventory[
        "txt_sha256"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

selected_hashes = set(
    representatives[
        "txt_sha256"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


expected_hashes_before_exclusions = (
    selected_hashes
    if PILOT_MODE
    else all_source_hashes
)


if not expected_hashes_before_exclusions:
    raise RuntimeError(
        "No hashes are expected for the current run scope."
    )


physical_paths = sorted(
    RECORD_ROOT.glob(
        "file_hash=*/part-*.parquet"
    )
)

physical_hashes = {
    path.parent.name.replace(
        "file_hash=",
        "",
        1,
    )
    for path in physical_paths
}


pass_hashes = set(
    manifest.loc[
        manifest["status_norm"].eq("PASS"),
        "txt_sha256",
    ]
) - {
    "",
}

fail_hashes = set(
    manifest.loc[
        manifest["status_norm"].eq("FAIL"),
        "txt_sha256",
    ]
) - {
    "",
}


# -----------------------------------------------------------------------------
# 09.03 Identify and approve exactly the known two unresolved hashes
# -----------------------------------------------------------------------------

unresolved_before_exclusions = sorted(
    expected_hashes_before_exclusions
    - (
        pass_hashes
        & physical_hashes
    )
)


if PILOT_MODE:

    if unresolved_before_exclusions:
        raise RuntimeError(
            "Pilot mode does not permit terminal exclusions. "
            f"Unresolved hashes: {len(unresolved_before_exclusions):,}"
        )

    approved_exclusion_hashes = set()

else:

    if len(unresolved_before_exclusions) != 2:
        raise RuntimeError(
            "This corrected full run expects exactly two known "
            "terminal exclusions, but found "
            f"{len(unresolved_before_exclusions):,} unresolved hashes."
        )

    approved_exclusion_hashes = set(
        unresolved_before_exclusions
    )


APPROVED_EXCLUSION_PATH = (
    REPORT_ROOT
    / "approved_terminal_exclusions.csv"
)


inventory_columns = [
    column
    for column in [
        "txt_sha256",
        "txt_file",
        "txt_size_bytes",
        "txt_mtime_ns_reference_only",
    ]
    if column in unique_inventory.columns
]


if approved_exclusion_hashes:

    approved_exclusions = (
        unique_inventory[
            inventory_columns
        ]
        .loc[
            unique_inventory[
                "txt_sha256"
            ]
            .astype(str)
            .isin(
                approved_exclusion_hashes
            )
        ]
        .drop_duplicates(
            "txt_sha256",
            keep="last",
        )
        .merge(
            manifest[
                [
                    column
                    for column in [
                        "txt_sha256",
                        "status",
                        "stage",
                        "records",
                        "decoded_rows",
                        "error",
                    ]
                    if column in manifest.columns
                ]
            ],
            on="txt_sha256",
            how="left",
            validate="one_to_one",
        )
    )

    approved_exclusions[
        "exclusion_status"
    ] = "APPROVED_TERMINAL_EXCLUSION"

    approved_exclusions[
        "exclusion_reason"
    ] = (
        "Source remained unprocessable after corrected full "
        "ROW V2 rebuild and is excluded from downstream use."
    )

    approved_exclusions[
        "parser_version"
    ] = PARSER_VERSION

    approved_exclusions[
        "decoder_version"
    ] = DECODER_VERSION

    approved_exclusions[
        "timestamp_contract_version"
    ] = TIMESTAMP_CONTRACT_VERSION

    approved_exclusions[
        "run_id"
    ] = RUN_ID

else:

    approved_exclusions = pd.DataFrame(
        columns=[
            *inventory_columns,
            "status",
            "stage",
            "records",
            "decoded_rows",
            "error",
            "exclusion_status",
            "exclusion_reason",
            "parser_version",
            "decoder_version",
            "timestamp_contract_version",
            "run_id",
        ]
    )


approved_exclusions.to_csv(
    APPROVED_EXCLUSION_PATH,
    index=False,
)


APPROVED_EXCLUSION_HASHES = set(
    approved_exclusions[
        "txt_sha256"
    ]
    .dropna()
    .astype(str)
)


effective_expected_hashes = (
    expected_hashes_before_exclusions
    - APPROVED_EXCLUSION_HASHES
)


# -----------------------------------------------------------------------------
# 09.04 Reconcile effective pipeline population
# -----------------------------------------------------------------------------

missing_expected_partitions = (
    effective_expected_hashes
    - physical_hashes
)

selected_without_manifest_pass = (
    effective_expected_hashes
    - pass_hashes
)

selected_pipeline_failures = (
    effective_expected_hashes
    & fail_hashes
)

unprocessed_source_hashes = (
    all_source_hashes
    - physical_hashes
    - APPROVED_EXCLUSION_HASHES
)

unexpected_partitions = (
    physical_hashes
    - all_source_hashes
)


# -----------------------------------------------------------------------------
# 09.05 Reconcile pairing manifest
# -----------------------------------------------------------------------------

def partition_hash(value):

    try:
        return (
            Path(
                str(value)
            )
            .parent
            .name
            .replace(
                "file_hash=",
                "",
                1,
            )
        )

    except Exception:
        return ""


pair_manifest[
    "txt_sha256_from_partition"
] = pair_manifest[
    "partition"
].map(
    partition_hash
)


pair_pass_hashes = set(
    pair_manifest.loc[
        pair_manifest[
            "status_norm"
        ].eq("PASS"),
        "txt_sha256_from_partition",
    ].astype(str)
) - {
    "",
}


pair_fail_hashes = set(
    pair_manifest.loc[
        pair_manifest[
            "status_norm"
        ].eq("FAIL"),
        "txt_sha256_from_partition",
    ].astype(str)
) - {
    "",
}


selected_without_pairing_pass = (
    effective_expected_hashes
    - pair_pass_hashes
)

selected_pairing_failures = (
    effective_expected_hashes
    & pair_fail_hashes
)


# -----------------------------------------------------------------------------
# 09.06 Validate canonical schema
# -----------------------------------------------------------------------------

required_columns = {
    "record_uid",
    "txt_sha256",
    "source_txt_file",
    "source_line_start",
    "source_line_end",
    "section",
    "payload_json",
    "engine_serial",
    "record_date_raw",
    "record_time_raw",
    "record_datetime_utc",
    "timestamp_source",
    "timestamp_format",
    "timestamp_quality",
    "timestamp_contract_version",
    "engine_status_word_1_raw",
    "engine_status_word_2_raw",
    "engine_status_word_3_raw",
    "engine_status_word_4_raw",
    "decoder_status",
    "decoder_version",
    "decoder_definition_sha256",
    "decoded_status_fields_json",
    "aircraft_identifier",
    "aircraft_identity_status",
    "cross_side_engine_serial",
    "cross_side_engine_resolution_status",
}


selected_partition_paths = [
    path
    for path in physical_paths
    if (
        path.parent.name.replace(
            "file_hash=",
            "",
            1,
        )
        in effective_expected_hashes
    )
]


schema_paths = (
    selected_partition_paths
    if PILOT_MODE
    else selected_partition_paths[:1000]
)


schema_rows = []


for index, path in enumerate(
    schema_paths,
    start=1,
):

    try:

        names = set(
            pq.ParquetFile(
                path
            )
            .schema_arrow
            .names
        )

        missing_columns = sorted(
            required_columns
            - names
        )

        schema_rows.append(
            {
                "partition":
                    str(path),

                "missing_columns":
                    json.dumps(
                        missing_columns
                    ),

                "status":
                    (
                        "PASS"
                        if not missing_columns
                        else "FAIL"
                    ),

                "error":
                    "",
            }
        )

    except Exception as exc:

        schema_rows.append(
            {
                "partition":
                    str(path),

                "missing_columns":
                    json.dumps([]),

                "status":
                    "READ_FAIL",

                "error":
                    repr(exc),
            }
        )


    if (
        index == 1
        or index % 250 == 0
        or index == len(schema_paths)
    ):
        print(
            f"Schema checked "
            f"{index:,}/{len(schema_paths):,}"
        )


schema_audit = pd.DataFrame(
    schema_rows,
    columns=[
        "partition",
        "missing_columns",
        "status",
        "error",
    ],
)


schema_audit_path = (
    REPORT_ROOT
    / (
        "pilot_schema_audit.csv"
        if PILOT_MODE
        else "full_schema_sample_audit.csv"
    )
)


schema_audit.to_csv(
    schema_audit_path,
    index=False,
)


schema_failures = int(
    schema_audit[
        "status"
    ].ne(
        "PASS"
    ).sum()
)


# -----------------------------------------------------------------------------
# 09.07 Summary
# -----------------------------------------------------------------------------

hash_failures = int(
    inventory[
        "hash_status"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("FAIL")
    .sum()
)


summary = {
    "run_id":
        RUN_ID,

    "run_scope":
        (
            "PILOT"
            if PILOT_MODE
            else "FULL_POPULATION_WITH_APPROVED_EXCLUSIONS"
        ),

    "pilot_mode":
        bool(
            PILOT_MODE
        ),

    "timestamp_contract_version":
        TIMESTAMP_CONTRACT_VERSION,

    "physical_txt_paths_rehashed":
        int(
            len(inventory)
        ),

    "hash_failures":
        hash_failures,

    "source_hashes_before_exclusions":
        int(
            len(
                expected_hashes_before_exclusions
            )
        ),

    "approved_terminal_exclusions":
        int(
            len(
                APPROVED_EXCLUSION_HASHES
            )
        ),

    "effective_expected_hashes":
        int(
            len(
                effective_expected_hashes
            )
        ),

    "pipeline_pass_hashes_in_effective_scope":
        int(
            len(
                pass_hashes
                & effective_expected_hashes
            )
        ),

    "pipeline_fail_hashes_in_effective_scope":
        int(
            len(
                selected_pipeline_failures
            )
        ),

    "physical_partitions_total":
        int(
            len(
                physical_hashes
            )
        ),

    "effective_partitions_present":
        int(
            len(
                physical_hashes
                & effective_expected_hashes
            )
        ),

    "missing_effective_partitions":
        int(
            len(
                missing_expected_partitions
            )
        ),

    "unprocessed_unapproved_source_hashes":
        int(
            len(
                unprocessed_source_hashes
            )
        ),

    "unexpected_partitions":
        int(
            len(
                unexpected_partitions
            )
        ),

    "effective_hashes_without_manifest_pass":
        int(
            len(
                selected_without_manifest_pass
            )
        ),

    "pairing_pass_hashes_in_effective_scope":
        int(
            len(
                pair_pass_hashes
                & effective_expected_hashes
            )
        ),

    "pairing_fail_hashes_in_effective_scope":
        int(
            len(
                selected_pairing_failures
            )
        ),

    "effective_hashes_without_pairing_pass":
        int(
            len(
                selected_without_pairing_pass
            )
        ),

    "schema_partitions_checked":
        int(
            len(
                schema_audit
            )
        ),

    "schema_failures":
        schema_failures,

    "approved_exclusion_manifest":
        str(
            APPROVED_EXCLUSION_PATH
        ),

    "output_root":
        str(
            OUTPUT_ROOT
        ),

    "schema_audit":
        str(
            schema_audit_path
        ),
}


summary_path = (
    REPORT_ROOT
    / (
        "pilot_run_summary.json"
        if PILOT_MODE
        else "full_run_summary.json"
    )
)


summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\nCELL 09 RECONCILIATION SUMMARY"
)

print(
    json.dumps(
        summary,
        indent=2,
    )
)


# -----------------------------------------------------------------------------
# 09.08 Blocking checks
# -----------------------------------------------------------------------------

blocking_errors = []


if summary["hash_failures"] != 0:
    blocking_errors.append(
        "Some TXT files could not be hashed."
    )

if summary["missing_effective_partitions"] != 0:
    blocking_errors.append(
        "Some effective expected hashes lack output partitions."
    )

if (
    summary[
        "effective_hashes_without_manifest_pass"
    ]
    != 0
):
    blocking_errors.append(
        "Some effective expected hashes are not PASS "
        "in the pipeline manifest."
    )

if (
    summary[
        "pipeline_fail_hashes_in_effective_scope"
    ]
    != 0
):
    blocking_errors.append(
        "The effective population contains pipeline failures."
    )

if (
    summary[
        "unprocessed_unapproved_source_hashes"
    ]
    != 0
):
    blocking_errors.append(
        "Source hashes remain unprocessed and are not "
        "approved terminal exclusions."
    )

if summary["unexpected_partitions"] != 0:
    blocking_errors.append(
        "Output contains partitions absent from the "
        "refreshed TXT inventory."
    )

if (
    summary[
        "effective_hashes_without_pairing_pass"
    ]
    != 0
):
    blocking_errors.append(
        "Some effective expected hashes are not PASS "
        "in the pairing manifest."
    )

if (
    summary[
        "pairing_fail_hashes_in_effective_scope"
    ]
    != 0
):
    blocking_errors.append(
        "The effective population contains pairing failures."
    )

if summary["schema_failures"] != 0:
    blocking_errors.append(
        "Some checked partitions failed the canonical schema contract."
    )


if not PILOT_MODE:

    if len(
        APPROVED_EXCLUSION_HASHES
    ) != 2:
        blocking_errors.append(
            "The full run requires exactly two documented "
            "terminal exclusions."
        )


if blocking_errors:

    print(
        "\nBLOCKING ERRORS"
    )

    for error in blocking_errors:
        print(
            " -",
            error,
        )

    raise RuntimeError(
        "Cell 09 effective-population reconciliation failed."
    )


# -----------------------------------------------------------------------------
# 09.09 Final status
# -----------------------------------------------------------------------------

if PILOT_MODE:

    print(
        "\nPASS: Pilot reconciliation completed."
    )

    print(
        "\nEXPANSION SETTINGS"
    )

    print(
        "PILOT_MODE = False"
    )

    print(
        "INITIALIZE_FULL_REBUILD = False"
    )

    print(
        "RESUME_REBUILD = True"
    )

else:

    if len(
        physical_hashes
        & effective_expected_hashes
    ) != len(
        effective_expected_hashes
    ):
        raise RuntimeError(
            "Physical partition count does not equal "
            "the effective expected population."
        )

    print(
        "\nPASS: Full-population reconciliation completed "
        "with two approved terminal exclusions."
    )

    print(
        {
            "source_hashes":
                len(
                    expected_hashes_before_exclusions
                ),

            "approved_exclusions":
                len(
                    APPROVED_EXCLUSION_HASHES
                ),

            "effective_canonical_partitions":
                len(
                    effective_expected_hashes
                ),
        }
    )

Schema checked 1/1,000
Schema checked 250/1,000
Schema checked 500/1,000
Schema checked 750/1,000
Schema checked 1,000/1,000

CELL 09 RECONCILIATION SUMMARY
{
  "run_id": "11001_ROW_V2_TIMESTAMP_CORRECTED_20260920",
  "run_scope": "FULL_POPULATION_WITH_APPROVED_EXCLUSIONS",
  "pilot_mode": false,
  "timestamp_contract_version": "SECTION_GMT_YYMMDD_HHMMSS_V1",
  "physical_txt_paths_rehashed": 108174,
  "hash_failures": 0,
  "source_hashes_before_exclusions": 107968,
  "approved_terminal_exclusions": 2,
  "effective_expected_hashes": 107966,
  "pipeline_pass_hashes_in_effective_scope": 107966,
  "pipeline_fail_hashes_in_effective_scope": 0,
  "physical_partitions_total": 107966,
  "effective_partitions_present": 107966,
  "missing_effective_partitions": 0,
  "unprocessed_unapproved_source_hashes": 0,
  "unexpected_partitions": 0,
  "effective_hashes_without_manifest_pass": 0,
  "pairing_pass_hashes_in_effective_scope": 107966,
  "pairing_fail_hashes_in_effective_scope": 0,
  "effective_h

In [23]:
# =============================================================================
# CELL 10 - Effective-population validation gate
#
# Validates:
# - 107,966 effective canonical partitions
# - exactly two approved terminal exclusions
# - no unapproved pipeline or pairing failures
# - complete schema and timestamp contracts
# - no pre-2000 or future emitted timestamps
# =============================================================================

import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 10.01 Required Cell 09 state
# -----------------------------------------------------------------------------

required_globals = [
    "APPROVED_EXCLUSION_HASHES",
    "APPROVED_EXCLUSION_PATH",
    "effective_expected_hashes",
    "expected_hashes_before_exclusions",
    "TIMESTAMP_CONTRACT_VERSION",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Cell 10 requires the completed replacement Cell 09. "
        f"Missing variables: {missing_globals}"
    )


if not APPROVED_EXCLUSION_PATH.is_file():
    raise FileNotFoundError(
        APPROVED_EXCLUSION_PATH
    )


approved_exclusions = pd.read_csv(
    APPROVED_EXCLUSION_PATH
)


approved_exclusion_hashes = set(
    approved_exclusions[
        "txt_sha256"
    ]
    .dropna()
    .astype(str)
)


if approved_exclusion_hashes != set(
    APPROVED_EXCLUSION_HASHES
):
    raise RuntimeError(
        "In-memory and persisted terminal exclusion sets do not match."
    )


# -----------------------------------------------------------------------------
# 10.02 Reload current manifests
# -----------------------------------------------------------------------------

manifest = (
    pd.read_parquet(
        MANIFEST_PATH
    )
    .drop_duplicates(
        "txt_sha256",
        keep="last",
    )
)

pair_manifest = (
    pd.read_parquet(
        PAIRING_MANIFEST_PATH
    )
    .drop_duplicates(
        "partition",
        keep="last",
    )
)


manifest["txt_sha256"] = (
    manifest["txt_sha256"]
    .fillna("")
    .astype(str)
    .str.strip()
)

manifest["status_norm"] = (
    manifest["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


pair_manifest["status_norm"] = (
    pair_manifest["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


source_selected_hashes = set(
    representatives[
        "txt_sha256"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


selected_hashes = (
    source_selected_hashes
    - approved_exclusion_hashes
)


pass_hashes = set(
    manifest.loc[
        manifest[
            "status_norm"
        ].eq("PASS"),
        "txt_sha256",
    ]
) - {
    "",
}


physical_paths = sorted(
    RECORD_ROOT.glob(
        "file_hash=*/part-*.parquet"
    )
)


partition_hashes = {
    path.parent.name.replace(
        "file_hash=",
        "",
        1,
    )
    for path in physical_paths
}


def pair_partition_hash(value):

    try:
        return (
            Path(
                str(value)
            )
            .parent
            .name
            .replace(
                "file_hash=",
                "",
                1,
            )
        )

    except Exception:
        return ""


pair_manifest[
    "txt_sha256_from_partition"
] = pair_manifest[
    "partition"
].map(
    pair_partition_hash
)


pair_pass_hashes = set(
    pair_manifest.loc[
        pair_manifest[
            "status_norm"
        ].eq("PASS"),
        "txt_sha256_from_partition",
    ].astype(str)
) - {
    "",
}


effective_pipeline_failures = manifest.loc[
    manifest[
        "status_norm"
    ].eq("FAIL")
    & manifest[
        "txt_sha256"
    ].isin(
        selected_hashes
    )
]


effective_pairing_failures = pair_manifest.loc[
    pair_manifest[
        "status_norm"
    ].eq("FAIL")
    & pair_manifest[
        "txt_sha256_from_partition"
    ].isin(
        selected_hashes
    )
]


# -----------------------------------------------------------------------------
# 10.03 Validation helpers
# -----------------------------------------------------------------------------

checks = []


def add_check(
    name,
    actual,
    expected,
    passed,
    severity="ERROR",
):

    checks.append(
        {
            "check":
                name,

            "actual":
                actual,

            "expected":
                expected,

            "status":
                (
                    "PASS"
                    if passed
                    else severity
                ),
        }
    )


expected_source_count = (
    1000
    if PILOT_MODE
    else len(
        unique_inventory
    )
)


expected_effective_count = (
    expected_source_count
    - len(
        approved_exclusion_hashes
        & source_selected_hashes
    )
)


add_check(
    "Source selected unique hashes",
    len(
        source_selected_hashes
    ),
    expected_source_count,
    len(
        source_selected_hashes
    ) == expected_source_count,
)

add_check(
    "Approved terminal exclusions",
    len(
        approved_exclusion_hashes
        & source_selected_hashes
    ),
    (
        0
        if PILOT_MODE
        else 2
    ),
    len(
        approved_exclusion_hashes
        & source_selected_hashes
    )
    == (
        0
        if PILOT_MODE
        else 2
    ),
)

add_check(
    "Effective selected unique hashes",
    len(
        selected_hashes
    ),
    expected_effective_count,
    len(
        selected_hashes
    ) == expected_effective_count,
)

add_check(
    "Manifest PASS covers effective hashes",
    len(
        pass_hashes
        & selected_hashes
    ),
    len(
        selected_hashes
    ),
    selected_hashes.issubset(
        pass_hashes
    ),
)

add_check(
    "Physical partitions cover effective hashes",
    len(
        partition_hashes
        & selected_hashes
    ),
    len(
        selected_hashes
    ),
    selected_hashes.issubset(
        partition_hashes
    ),
)

add_check(
    "No unexpected partitions",
    len(
        partition_hashes
        - all_source_hashes
    ),
    0,
    len(
        partition_hashes
        - all_source_hashes
    ) == 0,
)

add_check(
    "Unapproved pipeline failures",
    len(
        effective_pipeline_failures
    ),
    0,
    effective_pipeline_failures.empty,
)

add_check(
    "Pairing PASS covers effective hashes",
    len(
        pair_pass_hashes
        & selected_hashes
    ),
    len(
        selected_hashes
    ),
    selected_hashes.issubset(
        pair_pass_hashes
    ),
)

add_check(
    "Unapproved pairing failures",
    len(
        effective_pairing_failures
    ),
    0,
    effective_pairing_failures.empty,
)


# -----------------------------------------------------------------------------
# 10.04 Full effective-partition validation
# -----------------------------------------------------------------------------

required = {
    "record_uid",
    "txt_sha256",
    "source_txt_file",
    "source_line_start",
    "source_line_end",
    "section",
    "payload_json",
    "engine_serial",
    "record_date_raw",
    "record_time_raw",
    "record_datetime_utc",
    "timestamp_source",
    "timestamp_format",
    "timestamp_quality",
    "timestamp_contract_version",
    "engine_status_word_1_raw",
    "engine_status_word_2_raw",
    "engine_status_word_3_raw",
    "engine_status_word_4_raw",
    "decoder_status",
    "decoder_version",
    "decoder_definition_sha256",
    "decoded_status_fields_json",
    "aircraft_identifier",
    "aircraft_identity_status",
    "cross_side_engine_serial",
    "cross_side_engine_resolution_status",
}


effective_paths = [
    path
    for path in physical_paths
    if (
        path.parent.name.replace(
            "file_hash=",
            "",
            1,
        )
        in selected_hashes
    )
]


schema_failures = []
unreadable = []
empty = []
mixed_hash = []

uid_duplicates = 0
total_rows = 0
decoded_rows = 0
raw_sw_rows = 0
timestamp_rows = 0

timestamps_before_2000 = 0
timestamps_after_run_date = 0
timestamp_contract_mismatches = 0
ff_wrong_timestamp_source = 0
landing_wrong_timestamp_source = 0


for index, path in enumerate(
    effective_paths,
    start=1,
):

    try:

        parquet_file = pq.ParquetFile(
            path
        )

        names = set(
            parquet_file
            .schema_arrow
            .names
        )

        missing = sorted(
            required
            - names
        )

        if missing:

            schema_failures.append(
                {
                    "partition":
                        str(path),

                    "missing":
                        json.dumps(
                            missing
                        ),
                }
            )

            continue


        read_columns = [
            column
            for column in [
                "record_uid",
                "txt_sha256",
                "section",
                "decoder_status",
                "engine_status_word_1_raw",
                "engine_status_word_2_raw",
                "engine_status_word_3_raw",
                "engine_status_word_4_raw",
                "record_datetime_utc",
                "timestamp_source",
                "timestamp_contract_version",
            ]
            if column in names
        ]


        frame = pd.read_parquet(
            path,
            columns=read_columns,
        )


        if frame.empty:

            empty.append(
                str(path)
            )

            continue


        total_rows += len(
            frame
        )


        uid_duplicates += int(
            frame[
                "record_uid"
            ].duplicated().sum()
        )


        expected_hash = (
            path.parent.name.replace(
                "file_hash=",
                "",
                1,
            )
        )


        observed_hashes = set(
            frame[
                "txt_sha256"
            ]
            .dropna()
            .astype(str)
        )


        if observed_hashes != {
            expected_hash
        }:

            mixed_hash.append(
                str(path)
            )


        decoded_rows += int(
            frame[
                "decoder_status"
            ]
            .fillna("")
            .astype(str)
            .str.startswith(
                "DECODED"
            )
            .sum()
        )


        sw_columns = [
            column
            for column in read_columns
            if column.startswith(
                "engine_status_word_"
            )
        ]


        if sw_columns:

            raw_sw_rows += int(
                frame[
                    sw_columns
                ]
                .notna()
                .any(axis=1)
                .sum()
            )


        timestamps = pd.to_datetime(
            frame[
                "record_datetime_utc"
            ],
            errors="coerce",
            utc=True,
        )


        timestamp_rows += int(
            timestamps.notna().sum()
        )


        timestamps_before_2000 += int(
            timestamps.lt(
                MIN_VALID_TIMESTAMP
            ).sum()
        )


        timestamps_after_run_date += int(
            timestamps.gt(
                MAX_VALID_TIMESTAMP
            ).sum()
        )


        timestamp_contract_mismatches += int(
            (
                ~frame[
                    "timestamp_contract_version"
                ].eq(
                    TIMESTAMP_CONTRACT_VERSION
                )
            ).sum()
        )


        section_norm = (
            frame[
                "section"
            ]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.upper()
        )


        ff_rows = section_norm.eq(
            "FF_ATTRIBUTES"
        )


        landing_rows = section_norm.eq(
            "LANDING_STATISTICS"
        )


        ff_wrong_timestamp_source += int(
            (
                ff_rows
                & timestamps.notna()
                & ~frame[
                    "timestamp_source"
                ].eq(
                    "FF GMT Date + FF GMT Time"
                )
            ).sum()
        )


        landing_wrong_timestamp_source += int(
            (
                landing_rows
                & timestamps.notna()
                & ~frame[
                    "timestamp_source"
                ].eq(
                    "Landing GMT Date + Landing GMT Time"
                )
            ).sum()
        )


    except Exception as exc:

        unreadable.append(
            {
                "partition":
                    str(path),

                "error":
                    repr(exc),
            }
        )


    if (
        index == 1
        or index % 250 == 0
        or index == len(
            effective_paths
        )
    ):

        print(
            f"Validated "
            f"{index:,}/{len(effective_paths):,} "
            "effective partitions"
        )


# -----------------------------------------------------------------------------
# 10.05 Final checks
# -----------------------------------------------------------------------------

add_check(
    "Unreadable effective partitions",
    len(
        unreadable
    ),
    0,
    len(
        unreadable
    ) == 0,
)

add_check(
    "Empty effective partitions",
    len(
        empty
    ),
    0,
    len(
        empty
    ) == 0,
)

add_check(
    "Schema failures",
    len(
        schema_failures
    ),
    0,
    len(
        schema_failures
    ) == 0,
)

add_check(
    "Partition/hash mismatches",
    len(
        mixed_hash
    ),
    0,
    len(
        mixed_hash
    ) == 0,
)

add_check(
    "Duplicate record_uid within partitions",
    uid_duplicates,
    0,
    uid_duplicates == 0,
)

add_check(
    "Total parsed rows",
    total_rows,
    "> 0",
    total_rows > 0,
)

add_check(
    "Rows with raw status words",
    raw_sw_rows,
    "> 0",
    raw_sw_rows > 0,
)

add_check(
    "Rows with decoded status",
    decoded_rows,
    "> 0",
    decoded_rows > 0,
)

add_check(
    "Rows with parsed timestamps",
    timestamp_rows,
    "> 0",
    timestamp_rows > 0,
)

add_check(
    "Timestamps before 2000",
    timestamps_before_2000,
    0,
    timestamps_before_2000 == 0,
)

add_check(
    "Timestamps after execution date",
    timestamps_after_run_date,
    0,
    timestamps_after_run_date == 0,
)

add_check(
    "Timestamp contract mismatches",
    timestamp_contract_mismatches,
    0,
    timestamp_contract_mismatches == 0,
)

add_check(
    "FF rows with wrong timestamp source",
    ff_wrong_timestamp_source,
    0,
    ff_wrong_timestamp_source == 0,
)

add_check(
    "Landing rows with wrong timestamp source",
    landing_wrong_timestamp_source,
    0,
    landing_wrong_timestamp_source == 0,
)


# -----------------------------------------------------------------------------
# 10.06 Save validation artifacts
# -----------------------------------------------------------------------------

validation = pd.DataFrame(
    checks
)


validation.to_csv(
    REPORT_ROOT
    / "pilot_or_full_validation.csv",
    index=False,
)


if schema_failures:

    pd.DataFrame(
        schema_failures
    ).to_csv(
        REPORT_ROOT
        / "validation_schema_failures.csv",
        index=False,
    )


if unreadable:

    pd.DataFrame(
        unreadable
    ).to_csv(
        REPORT_ROOT
        / "validation_unreadable_partitions.csv",
        index=False,
    )


validation_summary = {
    "run_id":
        RUN_ID,

    "pilot_mode":
        bool(
            PILOT_MODE
        ),

    "source_selected_hashes":
        len(
            source_selected_hashes
        ),

    "approved_terminal_exclusions":
        len(
            approved_exclusion_hashes
        ),

    "effective_selected_hashes":
        len(
            selected_hashes
        ),

    "effective_partitions_validated":
        len(
            effective_paths
        ),

    "total_rows":
        total_rows,

    "raw_status_word_rows":
        raw_sw_rows,

    "decoded_rows":
        decoded_rows,

    "timestamp_rows":
        timestamp_rows,

    "timestamp_contract_version":
        TIMESTAMP_CONTRACT_VERSION,

    "validation_status":
        (
            "PASS"
            if not validation[
                "status"
            ].eq("ERROR").any()
            else "FAIL"
        ),
}


(
    REPORT_ROOT
    / "final_validation_summary.json"
).write_text(
    json.dumps(
        validation_summary,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    validation.to_string(
        index=False
    )
)


print(
    json.dumps(
        validation_summary,
        indent=2,
    )
)


blocking = validation.loc[
    validation[
        "status"
    ].eq("ERROR")
]


if not blocking.empty:

    raise RuntimeError(
        "VALIDATION FAILED. Review "
        "pilot_or_full_validation.csv and exception files."
    )


print(
    "PASS: Effective canonical ROW V2 population validated."
)


if PILOT_MODE:

    print(
        "Pilot outputs are ready for expansion."
    )

else:

    print(
        "FULL POPULATION VALIDATED:"
    )

    print(
        f"Source hashes: "
        f"{len(source_selected_hashes):,}"
    )

    print(
        f"Approved terminal exclusions: "
        f"{len(approved_exclusion_hashes):,}"
    )

    print(
        f"Effective canonical partitions: "
        f"{len(selected_hashes):,}"
    )

    print(
        "Outputs are ready for downstream notebooks."
    )

Validated 1/107,966 effective partitions
Validated 250/107,966 effective partitions
Validated 500/107,966 effective partitions
Validated 750/107,966 effective partitions
Validated 1,000/107,966 effective partitions
Validated 1,250/107,966 effective partitions
Validated 1,500/107,966 effective partitions
Validated 1,750/107,966 effective partitions
Validated 2,000/107,966 effective partitions
Validated 2,250/107,966 effective partitions
Validated 2,500/107,966 effective partitions
Validated 2,750/107,966 effective partitions
Validated 3,000/107,966 effective partitions
Validated 3,250/107,966 effective partitions
Validated 3,500/107,966 effective partitions
Validated 3,750/107,966 effective partitions
Validated 4,000/107,966 effective partitions
Validated 4,250/107,966 effective partitions
Validated 4,500/107,966 effective partitions
Validated 4,750/107,966 effective partitions
Validated 5,000/107,966 effective partitions
Validated 5,250/107,966 effective partitions
Validated 5,500/107,